# CAFA5 STRING Interactions

This notebook processes STRING database protein-protein interaction (PPI) data and adds it to the CAFA5 reasoning dataset.

## Configuration

**Important**: Update the paths in the configuration cell below to match your local setup.

In [ ]:
# ============================================================================
# CONFIGURATION - Update these paths to match your local setup
# ============================================================================
import os

# Base directory for CAFA5 dataset (HuggingFace format)
CAFA5_BASE_DIR = os.path.join("data", "cafa5_reasoning")  # Update this path

# STRING database files directory
STRING_DATA_DIR = os.path.join("data", "string")  # Update this path

# Output directory for processed files
OUTPUT_DIR = os.path.join("data", "processed")  # Update this path

# Paths to STRING database files
UNIPROT_STRING_MAPPING = os.path.join(STRING_DATA_DIR, "all_organisms.uniprot_2_string.tsv")
STRING_PROTEIN_INFO = os.path.join(STRING_DATA_DIR, "protein.info.v12.0.txt")
STRING_PROTEIN_LINKS_PARQUET = os.path.join(STRING_DATA_DIR, "parquet_output_duckdb", "protein_links_space_delimited.parquet")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration loaded. Update paths above if needed.")

## Setup Instructions

For HPC/cluster environments, run the following commands on a compute node:

In [2]:
python -m venv $SLURM_TMPDIR/bioreason_env
module load python
module load gcc arrow
module load postgresql

source $SLURM_TMPDIR/bioreason_env/bin/activate
pip install --no-index notebook duckdb polars-u64-idx datasets
jupyter notebook --no-browser --ip=$(hostname -f) --port=8888

Downloads (from https://string-db.org/cgi/download)

https://stringdb-downloads.org/download/protein.links.detailed.v12.0.txt.gz

https://stringdb-downloads.org/download/protein.info.v12.0.txt.gz

# Initial Analysis

Load both splits of the cafa5 from storage

In [14]:
import os
import polars as pl

# base directory
base_dir = CAFA5_BASE_DIR

# file paths
train_fp = os.path.join(base_dir, "train-00000-of-00001.parquet")
test_fp  = os.path.join(base_dir, "test-00000-of-00001.parquet")

# read train & tag
train_pl = pl.read_parquet(train_fp).with_columns([
    pl.lit(True).alias("is_train")
])

# read test & tag
test_pl = pl.read_parquet(test_fp).with_columns([
    pl.lit(False).alias("is_train")
])

# concatenate into one DataFrame
cafa5_pl = pl.concat([train_pl, test_pl], how="vertical")

# sanity check
print(cafa5_pl.schema)
print(cafa5_pl.head(3))




Schema([('protein_id', String), ('protein_names', String), ('protein_function', String), ('organism', String), ('length', Float64), ('subcellular_location', String), ('sequence', String), ('go_ids', List(String)), ('go_bp', List(String)), ('go_mf', List(String)), ('go_cc', List(String)), ('interpro_ids', List(String)), ('structure_path', String), ('is_train', Boolean)])
shape: (3, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ protein_i ┆ protein_n ┆ protein_f ┆ organism  ┆ … ┆ go_cc     ┆ interpro_ ┆ structure ┆ is_train │
│ d         ┆ ames      ┆ unction   ┆ ---       ┆   ┆ ---       ┆ ids       ┆ _path     ┆ ---      │
│ ---       ┆ ---       ┆ ---       ┆ str       ┆   ┆ list[str] ┆ ---       ┆ ---       ┆ bool     │
│ str       ┆ str       ┆ str       ┆           ┆   ┆           ┆ list[str] ┆ str       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ A0A0

Load the uniprot to string conversion table (from string database)

In [20]:
# load uniprot to string data

import polars as pl

mapping_path = UNIPROT_STRING_MAPPING
cols = ["species", "uniprot_ac|uniprot_id", "string_id", "identity", "bit_score"]

# Load and clean, skipping the bad header
uniprot_string_map_df = pl.read_csv(
    mapping_path,
    separator="\t",
    comment_prefix="#",
    has_header=False,
    new_columns=cols,
    ignore_errors=True
)

# Split the combined column into accession and ID, then drop the original
uniprot_string_map_df = (
    uniprot_string_map_df
    .with_columns(
        pl.col("uniprot_ac|uniprot_id")
          .str.split_exact("|", 1)
          .struct
          .rename_fields(["uniprot_ac", "uniprot_id"])
          .alias("split")
    )
    .unnest("split")
    .drop("uniprot_ac|uniprot_id")
)

# Preview
# print(uniprot_string_map_df.head(5))


Reads the string/name table from raw text to polars

In [ ]:
import polars as pl

# --- Configuration fallback: only set if not already defined ---
if 'protein_info_path' not in globals():
    protein_info_path = STRING_PROTEIN_INFO

# Define the actual column names (strip the leading '#')
cols = ["string_protein_id", "preferred_name", "protein_size", "annotation"]

# --- Load the uncompressed protein.info file, using our header names ---
protein_info_df = pl.read_csv(
    protein_info_path,
    separator="\t",
    has_header=False,   # file's first line isn't a usable header
    skip_rows=1,        # skip the '#...' header line
    new_columns=cols    # apply our cleaned column names
)

# Preview columns & first few rows
print("Columns:", protein_info_df.columns)
print(protein_info_df.head(5))


Columns: ['string_protein_id', 'preferred_name', 'protein_size', 'annotation']
shape: (5, 4)
┌───────────────────┬────────────────┬──────────────┬─────────────────────────────────┐
│ string_protein_id ┆ preferred_name ┆ protein_size ┆ annotation                      │
│ ---               ┆ ---            ┆ ---          ┆ ---                             │
│ str               ┆ str            ┆ i64          ┆ str                             │
╞═══════════════════╪════════════════╪══════════════╪═════════════════════════════════╡
│ 23.BEL05_00025    ┆ BEL05_00025    ┆ 429          ┆ Hypothetical protein; Incomple… │
│ 23.BEL05_00030    ┆ OEG73659.1     ┆ 203          ┆ 2-keto-4-pentenoate hydratase;… │
│ 23.BEL05_00035    ┆ OEG73660.1     ┆ 1140         ┆ Hybrid sensor histidine kinase… │
│ 23.BEL05_00040    ┆ acsA           ┆ 650          ┆ acetate--CoA ligase; Catalyzes… │
│ 23.BEL05_00045    ┆ OEG73662.1     ┆ 579          ┆ ATP-dependent helicase; Derive… │
└───────────────────┴──────

Brings the string ids into the cafa5 dataset, matching based on the uniprot id

In [21]:
# join uniprot cafa5 data with string id

import polars as pl

# 2) Prepare the mapping: only uniprot_ac → string_id
map_pl = uniprot_string_map_df.select(["uniprot_ac", "string_id"])

# 3) Join on protein_id ↔ uniprot_ac
joined = cafa5_pl.join(
    map_pl,
    left_on="protein_id",
    right_on="uniprot_ac",
    how="left"
)

cafa5_pl = joined.select(
    cafa5_pl.columns + ["string_id"]
)

# 5) Quick check
print(cafa5_pl.head(10))


shape: (10, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬──────────┬───────────┐
│ protein_i ┆ protein_n ┆ protein_f ┆ organism  ┆ … ┆ interpro_ ┆ structure ┆ is_train ┆ string_id │
│ d         ┆ ames      ┆ unction   ┆ ---       ┆   ┆ ids       ┆ _path     ┆ ---      ┆ ---       │
│ ---       ┆ ---       ┆ ---       ┆ str       ┆   ┆ ---       ┆ ---       ┆ bool     ┆ str       │
│ str       ┆ str       ┆ str       ┆           ┆   ┆ list[str] ┆ str       ┆          ┆           │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪══════════╪═══════════╡
│ A0A087X1C ┆ Putative  ┆ May be    ┆ Homo      ┆ … ┆ ["IPR0011 ┆ AF-A0A087 ┆ true     ┆ null      │
│ 5         ┆ cytochrom ┆ responsib ┆ sapiens   ┆   ┆ 28", "IPR ┆ X1C5-F1-m ┆          ┆           │
│           ┆ e P450    ┆ le for    ┆ (Human)   ┆   ┆ 017972",  ┆ odel_v4.c ┆          ┆           │
│           ┆ 2D7 (…    ┆ the met…  ┆           ┆   ┆ … "…      ┆ if       

Convert the String data file from txt to a parquet file

In [4]:
# # This cell efficiently converts the STRING data file to a Parquet file using DuckDB.


# import duckdb
# import os

# # Paths
# input_path = os.path.join(STRING_DATA_DIR, 'protein.links.detailed.v12.0.txt.gz')
# output_dir = os.path.join(STRING_DATA_DIR, 'parquet_output_duckdb')
# output_path = os.path.join(output_dir, 'protein_links_space_delimited.parquet')

# # Ensure output directory exists
# os.makedirs(output_dir, exist_ok=True)

# print("Starting conversion with DuckDB... This will be much faster.")

# # Connect to DuckDB
# con = duckdb.connect()

# # Set the number of threads (cores) DuckDB should use
# # You can set this to the number of physical cores you have (12 in your case)
# # or even a bit higher if you have hyperthreading, but 12 is a good starting point.
# con.execute("SET threads = 12;") 

# # This single command streams the 25.8 billion rows from the gzipped CSV
# # and writes it to a single, efficient Parquet file.
# # DuckDB handles all the memory management and parallel processing.
# # Note: The original file is tab-separated ('\t'), so we specify that.
# # We also assume the first row is a header. If not, set header=False.
# con.execute(f"""
#     COPY (
#         SELECT *
#         FROM read_csv('{input_path}', header=True, delim=' ', compression='gzip')
#     ) TO '{output_path}' (FORMAT 'PARQUET', CODEC 'ZSTD');
# """)

# print(f"Successfully converted the entire file to {output_path}")

# # Close the connection
# con.close()

(8 min) Copy the STRING data to slurm tempdir for faster processing

In [ ]:
# --- Prep: parallel-copy parquet to local Slurm tempdir with progress bar ---
import os
import multiprocessing as mp

# make sure you have tqdm installed: pip install --user tqdm
from tqdm import tqdm

# --- Configuration ---
input_parquet_path = STRING_PROTEIN_LINKS_PARQUET

def _copy_chunk(src, dst, offset, size):
    with open(src, "rb") as fsrc, open(dst, "r+b") as fdst:
        fsrc.seek(offset)
        fdst.seek(offset)
        fdst.write(fsrc.read(size))

# Top-level worker so multiprocessing can pickle it
def _copy_worker(args):
    src, dst, offset, size = args
    _copy_chunk(src, dst, offset, size)
    return None

def parallel_copy_with_progress(src: str, dst: str, chunk_size: int = 512 * 1024 * 1024):
    total_size = os.path.getsize(src)
    # Pre-allocate destination file
    with open(dst, "wb") as f:
        f.truncate(total_size)

    # Build all chunk tasks
    tasks = [
        (src, dst, offset, min(chunk_size, total_size - offset))
        for offset in range(0, total_size, chunk_size)
    ]

    # Copy in parallel, wrapping the iterator in tqdm to see progress
    with mp.Pool(mp.cpu_count()) as pool:
        for _ in tqdm(
            pool.imap_unordered(_copy_worker, tasks),
            total=len(tasks),
            desc="Copying chunks",
            unit="chunk"
        ):
            pass

# Determine Slurm temp directory and target path
slurm_tmpdir = os.environ.get("SLURM_TMPDIR", "/tmp")
local_parquet_path = os.path.join(slurm_tmpdir, os.path.basename(input_parquet_path))

print(f"Starting parallel copy to:\n  {local_parquet_path}\n(512 MB per chunk, {mp.cpu_count()} cores)…")
parallel_copy_with_progress(input_parquet_path, local_parquet_path)

# Reassign so downstream code reads from the local copy
input_parquet_path = local_parquet_path
print("Done. Now reading from local path:", input_parquet_path)


Copying
  /scratch/naimerja/BioReason2/PPI/Data/STRING_Full_Data/parquet_output_duckdb/protein_links_space_delimited.parquet
to
  /localscratch/naimerja.45617684.0/protein_links_space_delimited.parquet


Filter string data to only include rows with confidence >70% and which are related to a cafa5 protein

Takes ~15 minutues with 64 cpus and slow SLURM disk performance, for faster performance copy protein_links_space_delimited.parquet to SLURM_TEMPDIR with above cell

In [ ]:
import polars as pl

if 'input_parquet_path' not in globals():
    input_parquet_path = STRING_PROTEIN_LINKS_PARQUET

# Configuration: use all threads and a suitable streaming chunk size
pl.Config.set_streaming_chunk_size(1_000_000)  # Optional: tune chunk size for streaming

MIN_COMBINED_SCORE = 700

# Build CAFA5 ID list (assuming cafa5_pl is a Polars DataFrame already in memory)
string_ids = cafa5_pl["string_id"].drop_nulls().unique().to_list()
print(f"CAFA5 filter will use {len(string_ids)} unique string IDs")

# Lazy scan with optimized parallelism
lf = pl.scan_parquet(
    input_parquet_path,
    parallel="row_groups",   # parallelize over row groups to utilize all cores
    use_statistics=True      # enable Parquet page skipping by stats
    # low_memory=False is default (we prefer performance),
    # rechunk=False is default (avoid extra copying before collect)
)

# Apply filters lazily
lf = lf.filter(pl.col("combined_score") >= MIN_COMBINED_SCORE)
lf = lf.filter(pl.col("protein1").is_in(string_ids))

# Collect results with streaming (to pipeline reading & filtering)
print("Executing query in streaming mode...")
final_df = lf.collect(engine="streaming")  # or engine="streaming" on newer Polars

print("Done! Final dataframe has shape", final_df.shape)


CAFA5 filter will use 99821 unique string IDs
Executing query in streaming mode...
Done! Final dataframe has shape (2770689, 10)


In [13]:
final_df.head(10)

protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
str,str,i64,i64,i64,i64,i64,i64,i64,i64
"""52.CMC5_047490""","""52.CMC5_052310""",105,0,774,753,0,0,223,955
"""52.CMC5_047490""","""52.CMC5_047480""",718,0,0,0,0,0,0,718
"""52.CMC5_047490""","""52.CMC5_084210""",91,0,0,753,0,0,157,794
"""52.CMC5_047490""","""52.CMC5_002690""",105,163,773,753,0,0,223,961
"""52.CMC5_047490""","""52.CMC5_058910""",91,0,0,753,0,0,157,794
"""52.CMC5_047500""","""52.CMC5_025350""",0,0,770,0,0,0,0,770
"""52.CMC5_047500""","""52.CMC5_074250""",87,0,742,71,0,0,0,762
"""52.CMC5_047500""","""52.CMC5_032580""",0,0,734,0,0,0,69,741
"""52.CMC5_047510""","""52.CMC5_023720""",325,0,0,629,188,0,354,851


In [14]:
print("Estimated in-memory size:", final_df.estimated_size() / 1e6, "MB")

Estimated in-memory size: 394.270205 MB


Writes the filtered string dataframe to disk

In [15]:
output_path = os.path.join(OUTPUT_DIR, "filtered_protein_links.parquet")
final_df.write_parquet(output_path, compression="zstd")


Reads the filtered string dataframe

In [10]:
import polars as pl

# Path to the saved file
output_path = os.path.join(OUTPUT_DIR, "filtered_protein_links.parquet")

# Read the file
filtered_df = pl.read_parquet(output_path)

# Show shape and head
print("Shape:", filtered_df.shape)
print(filtered_df.tail())


Shape: (4044631, 10)
shape: (5, 10)
┌────────────┬────────────┬────────────┬────────┬───┬───────────┬──────────┬───────────┬───────────┐
│ protein1   ┆ protein2   ┆ neighborho ┆ fusion ┆ … ┆ experimen ┆ database ┆ textminin ┆ combined_ │
│ ---        ┆ ---        ┆ od         ┆ ---    ┆   ┆ tal       ┆ ---      ┆ g         ┆ score     │
│ str        ┆ str        ┆ ---        ┆ i64    ┆   ┆ ---       ┆ i64      ┆ ---       ┆ ---       │
│            ┆            ┆ i64        ┆        ┆   ┆ i64       ┆          ┆ i64       ┆ i64       │
╞════════════╪════════════╪════════════╪════════╪═══╪═══════════╪══════════╪═══════════╪═══════════╡
│ 1803814.AY ┆ 1803814.AY ┆ 0          ┆ 0      ┆ … ┆ 830       ┆ 445      ┆ 50        ┆ 908       │
│ K19_21110  ┆ K19_09730  ┆            ┆        ┆   ┆           ┆          ┆           ┆           │
│ 1803814.AY ┆ 1803814.AY ┆ 0          ┆ 0      ┆ … ┆ 831       ┆ 445      ┆ 0         ┆ 929       │
│ K19_21110  ┆ K19_13875  ┆            ┆        ┆   ┆  

replace protein2 string code with protein2 name

In [11]:
import polars as pl

# --- Replace protein2 IDs with their preferred names ---
# 1. Rename original protein2 to protein2_string
filtered_df = filtered_df.rename({"protein2": "protein2_string"})

# 2. Join to bring in the preferred_name from protein_info_df
filtered_df = filtered_df.join(
    protein_info_df.select(["string_protein_id", "preferred_name"]),
    left_on="protein2_string",
    right_on="string_protein_id",
    how="left"
)

# 3. Overwrite protein2 with the joined preferred_name
filtered_df = filtered_df.with_columns(
    pl.col("preferred_name").alias("protein2")
)

# 4. Drop any helper columns that were added
to_drop = [c for c in ["string_protein_id", "preferred_name"] if c in filtered_df.columns]
filtered_df = filtered_df.drop(to_drop)

# Now filtered_df.protein2 contains the preferred names,
# and the original IDs are in filtered_df.protein2_string.


In [12]:
filtered_df

protein1,protein2_string,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,protein2
str,str,i64,i64,i64,i64,i64,i64,i64,i64,str
"""52.CMC5_047490""","""52.CMC5_052310""",105,0,774,753,0,0,223,955,"""hutI"""
"""52.CMC5_047490""","""52.CMC5_047480""",718,0,0,0,0,0,0,718,"""AKT40592.1"""
"""52.CMC5_047490""","""52.CMC5_084210""",91,0,0,753,0,0,157,794,"""AKT44181.1"""
"""52.CMC5_047490""","""52.CMC5_002690""",105,163,773,753,0,0,223,961,"""hutU"""
"""52.CMC5_047490""","""52.CMC5_058910""",91,0,0,753,0,0,157,794,"""AKT41685.1"""
…,…,…,…,…,…,…,…,…,…,…
"""1803814.AYK19_21110""","""1803814.AYK19_09730""",0,0,0,95,830,445,50,908,"""rps24e"""
"""1803814.AYK19_21110""","""1803814.AYK19_13875""",0,0,266,93,831,445,0,929,"""rps4e"""
"""1803814.AYK19_21110""","""1803814.AYK19_17125""",67,0,0,86,831,412,0,903,"""rps13"""


In [15]:
print("CAFA5 column types:")
print(cafa5_pl.schema)

print("\nFiltered DF column types:")
print(filtered_df.schema)


CAFA5 column types:
Schema([('protein_id', String), ('protein_names', String), ('protein_function', String), ('organism', String), ('length', Float64), ('subcellular_location', String), ('sequence', String), ('go_ids', List(String)), ('go_bp', List(String)), ('go_mf', List(String)), ('go_cc', List(String)), ('interpro_ids', List(String)), ('structure_path', String), ('is_train', Boolean)])

Filtered DF column types:
Schema([('protein1', String), ('protein2_string', String), ('neighborhood', Int64), ('fusion', Int64), ('cooccurence', Int64), ('coexpression', Int64), ('experimental', Int64), ('database', Int64), ('textmining', Int64), ('combined_score', Int64), ('protein2', String)])


Groups filtered string df by protein 1 and then adds column with all associated protein 2 and another with all proteins 2 along with individual scores

In [13]:
import polars as pl

grouped_df = (
    filtered_df
    .group_by("protein1")
    .agg([
        # 1. List of all protein2 values per protein1
        pl.col("protein2").implode().alias("interaction_partners"),

        # 2. Struct of full interaction info, collected into list
        pl.struct([
            pl.col("protein2"),
            pl.col("neighborhood"),
            pl.col("fusion"),
            pl.col("cooccurence"),
            pl.col("coexpression"),
            pl.col("experimental"),
            pl.col("database"),
            pl.col("textmining"),
            pl.col("combined_score"),
        ]).implode().alias("full_interaction_info"),
    ])
)

print("Grouped shape:", grouped_df.shape)
print(grouped_df.head())


Grouped shape: (134653, 3)
shape: (5, 3)
┌──────────────────────────┬─────────────────────────────────┬─────────────────────────────────┐
│ protein1                 ┆ interaction_partners            ┆ full_interaction_info           │
│ ---                      ┆ ---                             ┆ ---                             │
│ str                      ┆ list[str]                       ┆ list[struct[9]]                 │
╞══════════════════════════╪═════════════════════════════════╪═════════════════════════════════╡
│ 284592.Q6BLV6            ┆ ["Q6BWV8_DEBHA", "Q6BRD8_DEBHA… ┆ [{"Q6BWV8_DEBHA",0,0,0,0,0,900… │
│ 9606.ENSP00000380872     ┆ ["CD3G", "CD3E"]                ┆ [{"CD3G",0,0,0,833,0,0,157,852… │
│ 284812.P0CG72            ┆ ["ubc15", "set1", … "rps1502"]  ┆ [{"ubc15",0,0,0,54,975,0,212,9… │
│ 10090.ENSMUSP00000023869 ┆ ["Rack1", "Eif3b", … "Ebna1bp2… ┆ [{"Rack1",0,0,0,86,766,0,142,8… │
│ 284812.Q09704            ┆ ["rcl1", "utp20", … "utp24"]    ┆ [{"rcl1",0,0,0,175,553,

In [16]:
cafa5_pl

protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,interpro_ids,structure_path,is_train
str,str,str,str,f64,str,str,list[str],list[str],list[str],list[str],list[str],str,bool
"""A0A087X1C5""","""Putative cytochrome P450 2D7 (…","""May be responsible for the met…","""Homo sapiens (Human)""",515.0,"""Membrane {ECO:0000305}; Multi-…","""MGLEALVPLAMIVAIFLLLVDLMHRHQRWA…","[""GO:0008152"", ""GO:0051716"", … ""GO:0016712""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0042178""]","[""GO:0003674"", ""GO:0003824"", … ""GO:0070330""]","[""GO:0005575"", ""GO:0110165"", … ""GO:0043231""]","[""IPR001128"", ""IPR017972"", … ""IPR050182""]","""AF-A0A087X1C5-F1-model_v4.cif""",true
"""A0A0B4J2F0""","""Protein PIGBOS1 (PIGB opposite…","""Plays a role in regulation of …","""Homo sapiens (Human)""",54.0,"""Mitochondrion outer membrane {…","""MFRRLTFAQLLFATVLGIAGGVYIFQPVFE…","[""GO:0048583"", ""GO:0050789"", … ""GO:0003674""]","[""GO:0008150"", ""GO:0050789"", … ""GO:1900101""]","[""GO:0003674"", ""GO:0005488"", ""GO:0005515""]","[""GO:0005575"", ""GO:0110165"", … ""GO:0005741""]",null,"""AF-A0A0B4J2F0-F1-model_v4.cif""",true
"""A0A0A7EPL0""","""E4 SUMO-protein ligase PIAL1 (…","""E4-type SUMO ligase that promo…","""Arabidopsis thaliana (Mouse-ea…",847.0,"""Nucleus {ECO:0000305}.""","""MVIPATSRFGFRAEFNTKEFQASCISLANE…","[""GO:0008152"", ""GO:0036211"", … ""GO:0019787""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0016925""]","[""GO:0003674"", ""GO:0003824"", … ""GO:0019789""]",null,"[""IPR004181"", ""IPR013083""]","""AF-A0A0A7EPL0-F1-model_v4.cif""",true
"""A0A0B4J1G0""","""Low affinity immunoglobulin ga…","""Receptor for the invariable Fc…","""Mus musculus (Mouse)""",249.0,"""Cell membrane {ECO:0000269|Pub…","""MWQLLLPTALVLTAFSGIQAGLQKAVVNLD…","[""GO:0006898"", ""GO:0051234"", … ""GO:0019770""]","[""GO:0008150"", ""GO:0002376"", … ""GO:0006898""]","[""GO:0003674"", ""GO:0060089"", … ""GO:0019770""]","[""GO:0005575"", ""GO:0110165"", … ""GO:0005886""]","[""IPR007110"", ""IPR036179"", … ""IPR003598""]","""AF-A0A0B4J1G0-F1-model_v4.cif""",true
"""A0A0B4J1N3""","""Protein GPR15LG (Protein GPR15…","""Highly cationic protein that h…","""Mus musculus (Mouse)""",78.0,"""Secreted {ECO:0000250|UniProtK…","""MRLLALSGLLCMLLLCFCIFSSEGRRHPAK…","[""GO:0009605"", ""GO:0051716"", … ""GO:0042379""]","[""GO:0008150"", ""GO:0040011"", … ""GO:0048247""]","[""GO:0003674"", ""GO:0098772"", … ""GO:0008009""]",null,"[""IPR031713""]","""AF-A0A0B4J1N3-F1-model_v4.cif""",true
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Q4A3R3""",null,"""May play roles in mucosal defe…","""Sus scrofa """,null,null,"""MGTSAVILEICLLLSQVLTTVSSTTQTEST…","[""None""]","[""None""]","[""None""]","[""None""]","[""IPR000859"", ""IPR035914"", … ""IPR017977""]","""AF-Q4A3R3-F1-model_v4.cif""",false
"""Q5E9L2""",null,"""Inhibits cell proliferation by…","""Bos taurus (cattle)""",null,null,"""MNWRFVELLYFLFVWGRISVQPSHQEPAAT…","[""None""]","[""None""]","[""None""]","[""None""]","[""IPR033237"", ""IPR020864""]","""AF-Q5E9L2-F1-model_v4.cif""",false
"""Q3T0K9""",null,"""In unstressed cells, promotes …","""Bos taurus (cattle)""",null,null,"""MNSKGQYPTQPTYPVQPPGNPVYPQTLHLP…","[""None""]","[""None""]","[""None""]","[""None""]","[""IPR022730""]","""AF-Q3T0K9-F1-model_v4.cif""",false


Connects the PPI data with the cafa5 data

In [22]:
merged_df = cafa5_pl.join(
    grouped_df,
    left_on="string_id",
    right_on="protein1",
    how="left"
)

# Only drop 'protein1' if it exists (e.g., when join didn't auto-drop it)
if "protein1" in merged_df.columns:
    merged_df = merged_df.drop("protein1")

print("Merged shape:", merged_df.shape)
print(merged_df.select(["string_id", "interaction_partners", "full_interaction_info"]).head())


Merged shape: (275402, 17)
shape: (5, 3)
┌──────────────────────────┬────────────────────────────────┬─────────────────────────────────┐
│ string_id                ┆ interaction_partners           ┆ full_interaction_info           │
│ ---                      ┆ ---                            ┆ ---                             │
│ str                      ┆ list[str]                      ┆ list[struct[9]]                 │
╞══════════════════════════╪════════════════════════════════╪═════════════════════════════════╡
│ null                     ┆ null                           ┆ null                            │
│ 9606.ENSP00000484893     ┆ null                           ┆ null                            │
│ 3702.A0A0A7EPL0          ┆ ["ESD4", "SUMO1", … "SAE1B-1"] ┆ [{"ESD4",0,0,0,0,107,0,811,823… │
│ 10090.ENSMUSP00000077873 ┆ ["Gm5150", "Vav1", … "Cd3g"]   ┆ [{"Gm5150",0,0,0,139,414,0,490… │
│ 10090.ENSMUSP00000136426 ┆ ["Gpr15"]                      ┆ [{"Gpr15",0,0,0,0,0,0,792,792}… │

In [23]:
# The percentage of rows in cafa5 WITHOUT an entry in string->uniprot table
sum(merged_df['string_id'].is_null())

44140

In [31]:
# The percent of rows in cafa5 WITHOUT interaction partners
sum(merged_df['interaction_partners'].is_null())/len(merged_df)*100

27.20786341420905

Saves the cafa5 data with the 2 new PPI columns to disk

In [25]:
output_path = os.path.join(OUTPUT_DIR, "cafa5_merged_with_interactions.parquet")

merged_df.write_parquet(output_path, compression="zstd")


In [1]:
import polars as pl

# Path to the saved file
merged_path = os.path.join(OUTPUT_DIR, "cafa5_merged_with_interactions.parquet")

# Read the merged DataFrame
merged_df = pl.read_parquet(merged_path)

# Preview shape and a few rows
print("Shape:", merged_df.shape)
merged_df.sample(9)

Shape: (275402, 17)


protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,interpro_ids,structure_path,is_train,string_id,interaction_partners,full_interaction_info
str,str,str,str,f64,str,str,list[str],list[str],list[str],list[str],list[str],str,bool,str,list[str],list[struct[9]]
"""Q7ZU74""","""Insulin receptor substrate 2a …",null,"""Danio rerio (Zebrafish) (Brach…",1032.0,null,"""MASPPLKGGPSLSNSNNIHSNNIRKHGYLK…","[""GO:0050896"", ""GO:0036293"", … ""GO:0008150""]","[""GO:0008150"", ""GO:0050896"", … ""GO:0036293""]",null,null,"[""IPR039011"", ""IPR002404"", … ""IPR001849""]","""AF-Q7ZU74-F1-model_v4.cif""",true,null,null,null
"""P06634""","""ATP-dependent RNA helicase DED…","""ATP-binding RNA helicase invol…","""Saccharomyces cerevisiae (stra…",604.0,"""Cytoplasm.""","""MAELSEQVQNLSINDNNENGYVPPHLRGKP…","[""None""]","[""None""]","[""None""]","[""None""]","[""IPR011545"", ""IPR044763"", … ""IPR014014""]","""AF-P06634-F1-model_v4.cif""",false,"""4932.YOR204W""","[""PSP2"", ""ATP16"", … ""NOC3""]","[{""PSP2"",0,0,0,94,817,0,69,832}, {""ATP16"",0,0,0,109,788,0,49,804}, … {""NOC3"",0,0,0,168,763,0,87,804}]"
"""P9WFF7""","""Decaprenyl diphosphate synthas…","""Catalyzes the sequential conde…","""Mycobacterium tuberculosis (st…",296.0,"""Cell membrane {ECO:0000269|Pub…","""MARDARKRTSSNFPQLPPAPDDYPTFPDTS…","[""GO:0008152"", ""GO:0016094"", … ""GO:0033850""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0016094""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0030145""]","[""GO:0005575"", ""GO:0110165"", … ""GO:0005886""]","[""IPR001441"", ""IPR018520"", ""IPR036424""]","""AF-P9WFF7-F1-model_v4.cif""",true,"""83332.Rv2361c""","[""idsA1"", ""idsB"", … ""grcC1""]","[{""idsA1"",71,0,622,91,0,0,669,880}, {""idsB"",71,0,631,93,0,0,325,761}, … {""grcC1"",71,0,643,94,0,0,814,936}]"
"""Q24151""","""Signal transducer and transcri…","""Might play a role in signal tr…","""Drosophila melanogaster (Fruit…",761.0,"""Cytoplasm {ECO:0000250}. Nucle…","""MSLWKRISSHVDCEQRMAAYYEEKGMLELR…","[""None""]","[""None""]","[""None""]","[""None""]","[""IPR008967"", ""IPR000980"", … ""IPR013799""]","""AF-Q24151-F1-model_v4.cif""",false,"""7227.FBpp0306651""","[""Su(var)205"", ""dpp"", … ""Ptp61F""]","[{""Su(var)205"",0,0,0,47,517,0,893,946}, {""dpp"",0,0,0,50,231,0,674,741}, … {""Ptp61F"",0,0,0,61,362,750,707,950}]"
"""Q9M1G6""","""Basic leucine zipper 25 (AtbZI…","""Transcription factor that bind…","""Arabidopsis thaliana (Mouse-ea…",403.0,"""Nucleus {ECO:0000255|PROSITE-P…","""MHIVFSVDDLTESFWPVPAPAPSPGSSSTP…","[""None""]","[""None""]","[""None""]","[""None""]","[""IPR020983"", ""IPR004827"", ""IPR046347""]","""AF-Q9M1G6-F1-model_v4.cif""",false,"""3702.Q9M1G6""","[""ABI3"", ""BZIP2"", … ""F1O13.3""]","[{""ABI3"",0,0,0,0,230,0,956,964}, {""BZIP2"",0,0,0,0,394,0,566,725}, … {""F1O13.3"",0,0,0,0,0,0,757,757}]"
"""P06399""","""Fibrinogen alpha chain [Cleave…","""Cleaved by the protease thromb…","""Rattus norvegicus (Rat)""",782.0,"""Secreted {ECO:0000250|UniProtK…","""MLSLRVACLILSLASTVWTADTGTTSEFIE…","[""None""]","[""None""]","[""None""]","[""None""]","[""IPR037579"", ""IPR036056"", … ""IPR020837""]","""AF-P06399-F1-model_v4.cif""",false,"""10116.ENSRNOP00000060007""","[""Mbl1"", ""Ambp"", … ""Ahsg""]","[{""Mbl1"",0,0,0,750,0,0,111,768}, {""Ambp"",0,0,0,909,0,0,223,926}, … {""Ahsg"",0,0,0,884,98,0,652,960}]"
"""A0A494C0Q6""","""Poly(A)-specific ribonuclease …",null,"""Homo sapiens (Human)""",581.0,"""Cytoplasm {ECO:0000256|ARBA:AR…","""MEIIRSNFKSNLHKVYQAIEEADFFAIDGE…","[""GO:0005654"", ""GO:0005622"", … ""GO:0005634""]",null,null,"[""GO:0005575"", ""GO:0110165"", … ""GO:0005634""]","[""IPR051181"", ""IPR012677"", … ""IPR036397""]","""AF-A0A494C0Q6-F1-model_v4.cif""",true,null,null,null
"""Q385H8""","""Uncharacterized protein""",null,"""Trypanosoma brucei brucei (str…",321.0,null,"""METSGPKTVPGTDLCHRVLYAAVVGDNAKG…","[""GO:0005622"", ""GO:0043229"", … ""GO:0043231""]",null,null,"[""GO:0005575"", ""GO:0110165"", … ""GO:0005634""]",null,"

In [2]:
merged_df.sample(10)

protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,interpro_ids,structure_path,is_train,string_id,interaction_partners,full_interaction_info
str,str,str,str,f64,str,str,list[str],list[str],list[str],list[str],list[str],str,bool,str,list[str],list[struct[9]]
"""Q10295""","""Poly(A) polymerase pla1 (PAP) …","""Polymerase that creates the 3'…","""Schizosaccharomyces pombe (str…",566.0,"""Nucleus {ECO:0000269|PubMed:10…","""MTTKQWGITPPISTAPATEQENALNTALIN…","[""None""]","[""None""]","[""None""]","[""None""]","[""IPR043519"", ""IPR011068"", … ""IPR014492""]","""AF-Q10295-F1-model_v4.cif""",false,"""284812.Q10295""","[""rna15"", ""iss1"", … ""gtr2""]","[{""rna15"",0,0,0,0,745,900,881,996}, {""iss1"",0,0,0,71,980,720,962,999}, … {""gtr2"",0,734,0,0,0,0,0,734}]"
"""A0PGB2""","""Telomerase-associated protein …","""Component of a CST-like subcom…","""Tetrahymena thermophila (strai…",622.0,"""Chromosome, telomere {ECO:0000…","""MEIEEDLNLKILEDVKKLYLQSFDYIKNGI…","[""GO:1990904"", ""GO:1902494"", … ""GO:0032991""]",null,null,"[""GO:0005575"", ""GO:0110165"", … ""GO:0005634""]",null,"""AF-A0PGB2-F1-model_v4.cif""",true,null,null,null
"""Q814X0""","""NADH-quinone oxidoreductase su…","""NDH-1 shuttles electrons from …","""Bacillus cereus (strain ATCC 1…",333.0,"""Cell membrane {ECO:0000255|HAM…","""MIETLLQSPSSWTNFFIFFGLAVLLLFAVL…","[""None""]","[""None""]","[""None""]","[""None""]","[""IPR001694"", ""IPR018086""]","""AF-Q814X0-F1-model_v4.cif""",false,"""226900.BC_5297""","[""nuoA"", ""BC_2838"", … ""BC_2022""]","[{""nuoA"",805,0,774,908,871,976,609,999}, {""BC_2838"",115,0,0,757,753,485,236,975}, … {""BC_2022"",0,0,0,50,871,681,197,964}]"
"""Q9NZC9""","""SWI/SNF-related matrix-associa…","""ATP-dependent annealing helica…","""Homo sapiens (Human)""",954.0,"""Nucleus {ECO:0000269|PubMed:19…","""MSLPLTEEQRKKIEENRQKALARRAEKLLA…","[""None""]","[""None""]","[""None""]","[""None""]","[""IPR010003"", ""IPR014001"", … ""IPR000330""]","""AF-Q9NZC9-F1-model_v4.cif""",false,"""9606.ENSP00000349823""","[""MUS81"", ""RECQL5"", … ""RPA1""]","[{""MUS81"",0,0,0,314,136,0,739,831}, {""RECQL5"",48,0,0,170,139,0,640,722}, … {""RPA1"",0,0,0,89,994,0,401,996}]"
"""Q9T072""","""Transcription factor bHLH25 (B…",null,"""Arabidopsis thaliana (Mouse-ea…",328.0,"""Nucleus {ECO:0000255|PROSITE-P…","""MSILSTRWFSEQEIEENSIIQQFHMNSIVG…","[""None""]","[""None""]","[""None""]","[""None""]","[""IPR054502"", ""IPR011598"", … ""IPR036638""]","""AF-Q9T072-F1-model_v4.cif""",false,"""3702.Q9T072""","[""F13M23.290""]","[{""F13M23.290"",0,0,0,0,0,0,759,759}]"
"""Q86NP2""","""Negative elongation factor A""","""Essential component of the NEL…","""Drosophila melanogaster (Fruit…",1251.0,"""Nucleus {ECO:0000269|PubMed:15…","""MANVRDSDTSLWLHNKLGTSNDSWINGSIC…","[""GO:0034243"", ""GO:0065007"", … ""GO:0005654""]","[""GO:0008150"", ""GO:0065007"", … ""GO:0034244""]",null,"[""GO:0005575"", ""GO:0032991"", … ""GO:0005634""]","[""IPR037517"", ""IPR052828""]","""AF-Q86NP2-F1-model_v4.cif""",true,"""7227.FBpp0303402""","[""ear"", ""NELF-B"", … ""Spt5""]","[{""ear"",0,0,0,0,0,720,0,720}, {""NELF-B"",0,0,0,104,918,800,810,996}, … {""Spt5"",0,0,0,65,881,720,619,986}]"
"""Q9C110""","""Meiotically up-regulated gene …","""Has a role in meiosis.""","""Schizosaccharomyces pombe (str…",105.0,"""Cytoplasm {ECO:0000269|PubMed:…","""MYPKIVARCMIVITKNRESERNVLICYRNV…","[""GO:0005737"", ""GO:0005829"", … ""GO:0110165""]",null,null,"[""GO:0005575"", ""GO:0110165"", … ""GO:0005622""]",null,"""AF-Q9C110-F1-model_v4.cif""",true,"""284812.Q9C110""",null,null
"""P63142""","""Potassium voltage-gated channe…","""Voltage-gated potassium channe…","""Rattus norvegicus (Rat)""",499.0,"""Cell membrane {ECO:0000269|Pub…","""MTVATGDPVDEAAALPGHPQDTYDPEADHE…","[""None""]","[""None""]","[""None""]","[""None""]","[""IPR000210"", ""IPR005821"", … ""IPR027359""]","""AF-P63142-F1-model_v4.cif""",false,"""10116.ENSRNOP00000075852""","[""Kcna6"", ""Kcnab2"", … ""LOC687780""]"

In [3]:
from datasets import Dataset

# If merged_df is a Polars DataFrame:
train_pd = (
    merged_df
    .filter(pl.col("is_train") == True)
    .drop("is_train")
    .to_pandas()
)
test_pd = (
    merged_df
    .filter(pl.col("is_train") == False)
    .drop("is_train")
    .to_pandas()
)

train_hf = Dataset.from_pandas(train_pd, preserve_index=False)
test_hf  = Dataset.from_pandas(test_pd,  preserve_index=False)


In [6]:
train_hf

Dataset({
    features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'interpro_ids', 'structure_path', 'string_id', 'interaction_partners', 'full_interaction_info'],
    num_rows: 133538
})

In [7]:
from datasets import DatasetDict
import os

# --- Paths ---
base_hf_dir = "/scratch/naimerja/BioReason2/PPI/Data/HuggingFace"
# Save under the “cafa5_reasoning” folder so it parallels the existing structure
save_dir = os.path.join(base_hf_dir, "cafa5_reasoning")

# Ensure the directory exists
os.makedirs(save_dir, exist_ok=True)

# --- Combine and save ---
cafa5_ds = DatasetDict({
    "train": train_hf,
    "test":  test_hf,
})

cafa5_ds.save_to_disk(save_dir)

print(f"Hugging Face splits saved to: {save_dir}")


Saving the dataset (0/2 shards):   0%|          | 0/133538 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/141864 [00:00<?, ? examples/s]

Hugging Face splits saved to: /scratch/naimerja/BioReason2/PPI/Data/HuggingFace/cafa5_reasoning
